# StyleBot — Entrega 4 final

Notebook integrado para comprobar el flujo completo de StyleBot sin volver a entrenar los modelos finales.

**Responsable de integración:** Krysto Alberto Adan Jauregui Zúñiga.

Esta versión ya carga y analiza los resultados comprobados de la Entrega 3. También detecta y conecta automáticamente los módulos finales de inferencia CNN, simulación y comparación global conforme sean publicados por el equipo.

## 1. Ejecución en Google Colab

Ejecutar todas las celdas de arriba hacia abajo. La primera celda clona o actualiza el repositorio. Los componentes todavía no entregados se reportan como pendientes con un mensaje claro; no se inventan resultados ni se reentrenan modelos.

In [ ]:
# Preparación reproducible
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/krysto9872/StyleBot.git"
REPO_COLAB = Path("/content/StyleBot")

if Path("/content").exists():
    if REPO_COLAB.exists():
        subprocess.check_call(["git", "-C", str(REPO_COLAB), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_COLAB)])
    RAIZ = REPO_COLAB
else:
    RAIZ = Path.cwd()

os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

try:
    import gymnasium  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gymnasium"])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SEMILLA = 42
SEMILLA_DEMO = 2026
np.random.seed(SEMILLA)

print("Raíz del repositorio:", RAIZ.resolve())
if (RAIZ / ".git").exists():
    commit = subprocess.check_output(["git", "-C", str(RAIZ), "rev-parse", "--short", "HEAD"], text=True).strip()
    print("Commit utilizado:", commit)

## 2. Protocolo común

StyleBot recibe una prenda, la CNN congelada predice su categoría y un agente de aprendizaje por refuerzo la lleva al estante correspondiente. La etiqueta real se utiliza únicamente para evaluar; la política recibe la categoría predicha por la CNN.

In [ ]:
# Carga del protocolo oficial de la Entrega 3
RUTA_PROTOCOLO = RAIZ / "datos/entrega3/protocolo_entrega3.json"
with RUTA_PROTOCOLO.open(encoding="utf-8") as archivo:
    protocolo = json.load(archivo)

CATEGORIAS = protocolo["categorias"]
ESTANTES = {int(k): tuple(v) for k, v in protocolo["estantes"].items()}
ACCIONES = {int(k): v for k, v in protocolo["acciones"].items()}

resumen_protocolo = pd.DataFrame({
    "elemento": ["Mapa", "Estados", "Acciones", "Prendas de demostración", "Semilla"],
    "valor": [
        " x ".join(map(str, protocolo["tamano_mapa"])),
        protocolo["numero_estados"],
        protocolo["numero_acciones"],
        protocolo["prendas_demo"],
        protocolo["semilla_demo"],
    ],
})
display(resumen_protocolo)
display(pd.DataFrame({"indice": range(10), "categoria": CATEGORIAS, "estante": [ESTANTES[i] for i in range(10)]}))

## 3. Archivos y rutas comunes

La siguiente comprobación permite identificar de inmediato qué componente falta. Los modelos guardados se cargan para inferencia o evaluación; nunca se entrenan al importar este notebook.

In [ ]:
RUTAS = {
    "modelo_cnn": RAIZ / "modelos/stylebot_cnn_final.keras",
    "tabla_qlearning": RAIZ / "modelos/entrega3/q_table_qlearning.npz",
    "tabla_dynaq": RAIZ / "modelos/entrega3/q_table_dynaq.npz",
    "predicciones_50": RAIZ / "datos/entrega3/predicciones_50_prendas.csv",
    "comparacion_agentes": RAIZ / "resultados/entrega3/comparacion_agentes.csv",
    "simulacion_50": RAIZ / "resultados/entrega3/simulacion_50_prendas_rl.csv",
    "analisis_fallos": RAIZ / "resultados/entrega3/analisis_fallos.csv",
    "inferencia_e4": RAIZ / "src/entrega4/inferencia_cnn.py",
    "simulacion_e4": RAIZ / "src/entrega4/simulacion_final.py",
    "comparacion_global": RAIZ / "resultados/entrega4/tabla_comparacion_global.csv",
}

estado_archivos = pd.DataFrame([
    {"componente": nombre, "existe": ruta.exists(), "ruta": str(ruta.relative_to(RAIZ))}
    for nombre, ruta in RUTAS.items()
])
display(estado_archivos)

## 4. Resultados comprobados de la Entrega 3

Estos archivos constituyen la línea base ya publicada. Se muestran para conservar trazabilidad mientras se incorporan las funciones finales de la Entrega 4.

In [ ]:
def cargar_csv(ruta, requerido=True):
    ruta = Path(ruta)
    if not ruta.exists():
        mensaje = f"No se encontró: {ruta.relative_to(RAIZ) if ruta.is_relative_to(RAIZ) else ruta}"
        if requerido:
            raise FileNotFoundError(mensaje)
        print("Pendiente —", mensaje)
        return None
    tabla = pd.read_csv(ruta)
    print(f"Cargado: {ruta.name} ({len(tabla)} filas)")
    return tabla

predicciones = cargar_csv(RUTAS["predicciones_50"])
comparacion_agentes = cargar_csv(RUTAS["comparacion_agentes"])
simulacion_50 = cargar_csv(RUTAS["simulacion_50"])
analisis_fallos = cargar_csv(RUTAS["analisis_fallos"])

predicciones["es_error_bool"] = predicciones["es_error"].astype(str).str.lower().eq("true")
aciertos_cnn = int((~predicciones["es_error_bool"]).sum())
errores_cnn = int(predicciones["es_error_bool"].sum())

print(f"CNN: {aciertos_cnn}/{len(predicciones)} clasificaciones correctas ({aciertos_cnn / len(predicciones):.1%}).")
print(f"Errores de percepción: {errores_cnn}/{len(predicciones)}.")
display(predicciones.head(10))
display(simulacion_50)

In [ ]:
# Comparación visual de Aleatorio, Q-Learning y Dyna-Q
orden = ["Aleatorio", "Q-Learning", "Dyna-Q"]
tabla_agentes = comparacion_agentes.copy()
tabla_agentes["agente"] = pd.Categorical(tabla_agentes["agente"], orden, ordered=True)
tabla_agentes = tabla_agentes.sort_values("agente")

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
colores = ["#9ca3af", "#2563eb", "#16a34a"]

ejes[0].bar(tabla_agentes["agente"], tabla_agentes["tasa_exito_prendas"] * 100, color=colores)
ejes[0].set_title("Tasa de éxito")
ejes[0].set_ylabel("Éxito (%)")
ejes[0].set_ylim(0, 100)

ejes[1].bar(tabla_agentes["agente"], tabla_agentes["recompensa_promedio"], color=colores)
ejes[1].set_title("Recompensa promedio")

ejes[2].bar(tabla_agentes["agente"], tabla_agentes["pasos_promedio"], color=colores)
ejes[2].set_title("Pasos promedio")

for eje in ejes:
    eje.tick_params(axis="x", rotation=20)
    eje.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

display(tabla_agentes[["agente", "tasa_exito_prendas", "recompensa_promedio", "pasos_promedio"]])

## 5. Inferencia CNN final

Cuando exista src/entrega4/inferencia_cnn.py se importará predecir_prenda. La función debe devolver índice, categoría, confianza e imagen procesada. Mientras el archivo no exista, el notebook continúa y muestra el pendiente.

In [ ]:
predecir_prenda = None
resultado_cnn = None

if RUTAS["inferencia_e4"].exists():
    importlib.invalidate_caches()
    from src.entrega4.inferencia_cnn import predecir_prenda

    from tensorflow.keras.datasets import fashion_mnist
    (_, _), (x_prueba, y_prueba) = fashion_mnist.load_data()
    resultado_cnn = predecir_prenda(x_prueba[0])

    campos_cnn = {"indice_categoria", "categoria", "confianza", "imagen_procesada"}
    faltantes_cnn = campos_cnn - set(resultado_cnn)
    assert not faltantes_cnn, f"Faltan campos de CNN: {sorted(faltantes_cnn)}"
    assert resultado_cnn["categoria"] in CATEGORIAS
    assert 0 <= float(resultado_cnn["confianza"]) <= 1

    print("Inferencia CNN validada:")
    display(pd.DataFrame([{k: v for k, v in resultado_cnn.items() if k != "imagen_procesada"}]))
else:
    print("Pendiente: Juan José debe publicar src/entrega4/inferencia_cnn.py")

## 6. Simulación final Q-Learning y Dyna-Q

Cuando exista src/entrega4/simulacion_final.py se importará simular_acomodo. La prueba utiliza una tabla Q guardada y no permite modificarla ni volver a entrenar.

In [ ]:
simular_acomodo = None
resultado_simulacion = None

if RUTAS["simulacion_e4"].exists():
    importlib.invalidate_caches()
    from src.entrega4.simulacion_final import simular_acomodo

    categoria_prueba = resultado_cnn["categoria"] if resultado_cnn else CATEGORIAS[0]
    resultado_simulacion = simular_acomodo(categoria_prueba, agente="dyna_q", semilla=SEMILLA_DEMO)

    campos_rl = {"agente", "categoria", "estante", "trayectoria", "acciones", "pasos", "recompensa", "exito", "tipo_fallo"}
    faltantes_rl = campos_rl - set(resultado_simulacion)
    assert not faltantes_rl, f"Faltan campos de simulación: {sorted(faltantes_rl)}"
    assert resultado_simulacion["categoria"] in CATEGORIAS
    assert int(resultado_simulacion["pasos"]) >= 0

    print("Simulación validada:")
    display(pd.DataFrame([{k: v for k, v in resultado_simulacion.items() if k not in {"trayectoria", "acciones"}}]))
else:
    print("Pendiente: Luis Ángel debe publicar src/entrega4/simulacion_final.py")

## 7. Flujo integrado CNN → estante → agente

Esta función representa el comportamiento final de StyleBot. Recibe una imagen y devuelve la percepción de la CNN junto con el resultado de la política seleccionada.

In [ ]:
def ejecutar_stylebot(imagen, agente="dyna_q", semilla=SEMILLA_DEMO):
    if predecir_prenda is None:
        raise RuntimeError("No está disponible src/entrega4/inferencia_cnn.py")
    if simular_acomodo is None:
        raise RuntimeError("No está disponible src/entrega4/simulacion_final.py")

    percepcion = predecir_prenda(imagen)
    simulacion = simular_acomodo(percepcion["categoria"], agente=agente, semilla=semilla)
    return {
        "categoria": percepcion["categoria"],
        "indice_categoria": percepcion["indice_categoria"],
        "confianza": float(percepcion["confianza"]),
        "estante": simulacion["estante"],
        "agente": simulacion["agente"],
        "trayectoria": simulacion["trayectoria"],
        "acciones": simulacion["acciones"],
        "pasos": simulacion["pasos"],
        "recompensa": simulacion["recompensa"],
        "exito": simulacion["exito"],
        "tipo_fallo": simulacion["tipo_fallo"],
    }

if predecir_prenda is not None and simular_acomodo is not None:
    resultado_integrado = ejecutar_stylebot(x_prueba[0], agente="dyna_q")
    print("Flujo completo validado.")
    display(pd.DataFrame([{k: v for k, v in resultado_integrado.items() if k not in {"trayectoria", "acciones"}}]))
else:
    print("El flujo está preparado y se activará cuando estén disponibles los dos módulos finales.")

## 8. Comparación global

La tabla final de Ángel Uriel debe distinguir accuracy, F1, Precision@5, tasa de éxito y recompensa. No se comparan directamente métricas con significados diferentes.

In [ ]:
comparacion_global = cargar_csv(RUTAS["comparacion_global"], requerido=False)

if comparacion_global is not None:
    columnas_requeridas = {"bloque", "modelo", "metrica_principal", "valor", "tiempo", "tamano", "observacion"}
    faltantes_global = columnas_requeridas - set(comparacion_global.columns)
    assert not faltantes_global, f"Faltan columnas en la comparación global: {sorted(faltantes_global)}"
    display(comparacion_global)
else:
    print("Pendiente: Ángel Uriel debe publicar resultados/entrega4/tabla_comparacion_global.csv")
    print("Mientras tanto se conserva la comparación verificada de los tres agentes:")
    display(tabla_agentes[["agente", "tasa_exito_prendas", "recompensa_promedio", "pasos_promedio"]])

## 9. Demostración de 50 prendas

La línea base publicada procesó 50 prendas con la semilla 2026. En la versión final se repetirá la demostración con las funciones de Entrega 4 y se separarán los fallos de percepción de los fallos de política.

In [ ]:
assert len(predicciones) == 50, "La demostración debe contener exactamente 50 prendas."

resumen_demo = simulacion_50[[
    "agente", "recompensa_total", "exito_prendas", "pasos",
    "correctas", "incorrectas", "fallos_cnn", "fallos_politica"
]].copy()
resumen_demo["exito_porcentaje"] = resumen_demo["exito_prendas"] * 100

display(resumen_demo)
display(analisis_fallos)

print(f"Resultado CNN comprobado: {aciertos_cnn} aciertos y {errores_cnn} error(es) en 50 prendas.")

## 10. Validación automática del estado final

In [ ]:
OBLIGATORIOS_FINALES = {
    **RUTAS,
    "informe_pdf": RAIZ / "documentacion/ProyectoFinal_StyleBot.pdf",
    "presentacion": RAIZ / "presentacion/StyleBot_Final.pptx",
}

validacion_final = pd.DataFrame([
    {"componente": nombre, "existe": ruta.exists(), "ruta": str(ruta.relative_to(RAIZ))}
    for nombre, ruta in OBLIGATORIOS_FINALES.items()
])
display(validacion_final)

pendientes = validacion_final.loc[~validacion_final["existe"], "componente"].tolist()
if pendientes:
    print("Integración todavía pendiente:", ", ".join(pendientes))
else:
    print("✅ Todos los componentes obligatorios están disponibles.")

assert RUTAS["modelo_cnn"].stat().st_size > 0
assert RUTAS["tabla_qlearning"].stat().st_size > 0
assert RUTAS["tabla_dynaq"].stat().st_size > 0
assert aciertos_cnn + errores_cnn == 50
assert set(tabla_agentes["agente"].astype(str)) == {"Aleatorio", "Q-Learning", "Dyna-Q"}
print("✅ La base técnica ya publicada fue validada correctamente.")

## 11. Conclusiones preliminares

- La base reproducible de la Entrega 3 está disponible: modelo CNN, tablas Q, protocolo y resultados de 50 prendas.
- La CNN registró 49 aciertos de 50 en la demostración publicada; el único fallo corresponde a percepción.
- Dyna-Q obtuvo el mejor resultado promedio entre los agentes evaluados: 93.6 % de éxito por prenda, recompensa promedio de -27.80 y 78.64 pasos promedio.
- Q-Learning alcanzó 92.4 % de éxito, mientras el agente aleatorio obtuvo 1.6 %.
- El notebook ya está preparado para importar las funciones finales sin reentrenar modelos.
- Las conclusiones definitivas se actualizarán después de recibir la comparación global y ejecutar el flujo final desde una sesión limpia.

## Fuentes

- Sutton, R. S. y Barto, A. G. Reinforcement Learning: An Introduction (2.ª ed.). http://incompleteideas.net/book/the-book-2nd.html
- Watkins, C. J. C. H. y Dayan, P. (1992). Q-learning. https://www.gatsby.ucl.ac.uk/~dayan/papers/cjch.pdf
- Gymnasium. Documentación oficial: https://gymnasium.farama.org/
- TensorFlow. Fashion-MNIST: https://www.tensorflow.org/api_docs/python/tf/keras/datasets/fashion_mnist/load_data